# MerLin release 0.4 highlights

We are presenting the new features and changes introduced in MerLin 0.4.x. 

## 0. Imports

In [ ]:
import merlin as ml
import perceval as pcvl
import torch

## 1. New features

Here is a brief overview of the main new features.

### 1.1 ``ReservoiClassifier``

...

For more details, checkout the [partial measurement documentation](../quantum_expert_area/partial_measurement.rst) from which we present the following example.

In [9]:
from merlin import CircuitBuilder, QuantumLayer
from merlin.core.computation_space import ComputationSpace
from merlin.measurement.strategies import MeasurementStrategy

# Minimal builder-based circuit definition.
builder = CircuitBuilder(n_modes=4)
builder.add_entangling_layer(trainable=True, name="U1")

# Partial measurement strategy (measure modes 0 and 1).
strategy = MeasurementStrategy.partial(
    modes=[0, 1],
    computation_space=ComputationSpace.FOCK,
)

layer = QuantumLayer(
    builder=builder,
    n_photons=2,
    measurement_strategy=strategy,
    return_object=True,
)

output = layer()

print(layer)
print(f" - Output type: {type(output)}")
print("--------- AMPLITUDES (unmeasured modes) ---------")
print(f" - Amplitudes = {output.amplitudes}")
print(f"\n - Amplitudes to tensor = {[amp.tensor for amp in output.amplitudes]}")
print(f"\n - Measured modes = {output.measured_modes}")
print(f"\n --------- PROBABILITIES (measured modes) ---------")
print(f"\n - Probabilities (tensor) = {output.probabilities}")

QuantumLayer(custom_circuit, modes=4, input_size=0, output_size=6)
 - Output type: <class 'merlin.core.partial_measurement.PartialMeasurement'>
--------- AMPLITUDES (unmeasured modes) ---------
 - Amplitudes = [StateVector(tensor=tensor([[ 0.0074-0.3472j,  0.3342+0.4957j, -0.1950-0.6957j]],
       grad_fn=<WhereBackward0>), n_modes=2, n_photons=2, _normalized=False), StateVector(tensor=tensor([[-0.7840+0.1447j,  0.6014-0.0526j]], grad_fn=<WhereBackward0>), n_modes=2, n_photons=1, _normalized=False), StateVector(tensor=tensor([[0.4434+0.8963j]], grad_fn=<WhereBackward0>), n_modes=2, n_photons=0, _normalized=False), StateVector(tensor=tensor([[-0.2238+0.7172j,  0.6204+0.2248j]], grad_fn=<WhereBackward0>), n_modes=2, n_photons=1, _normalized=False), StateVector(tensor=tensor([[0.5153-0.8570j]], grad_fn=<WhereBackward0>), n_modes=2, n_photons=0, _normalized=False), StateVector(tensor=tensor([[-0.9923-0.1241j]], grad_fn=<WhereBackward0>), n_modes=2, n_photons=0, _normalized=False)]

 - Ampl

## 2. Breaking changes
Here are features that are now completly removed from v.0.4.

### 2.1 The ``no_bunching`` flag is now removed

The ``no_bunching`` flag used in version 0.2 and 0.2 in many functions (QuantumLayer and kernels definitions) was deprecated since version 0.3.0 and is now removed. The new way of deciding to use the unbunched or full Fock computation space is with the ``ComputationSpace`` object in the ``MeasurementStrategy``. Here we present the two ways to define a ``QuantumLayer`` with the equivalent of settng the ``no_bunching`` flag to True or False.

In [ ]:
# Define the basic interferometer
circuit = ml.CircuitBuilder(n_modes=3)
circuit.add_entangling_layer()
circuit.add_angle_encoding([0, 1])
circuit.add_entangling_layer()

Old flag, breaking change

In [ ]:
# # Unbunched space
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     no_bunching=True
#     )

# # Full Fock space
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     no_bunching=False
#     )

Equivalents of the flag

In [ ]:
# Equivalent of no_bunching=True
qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.UNBUNCHED,
    ),
)
print(qlayer.computation_space)

## Or, because the unbunched space is applied by default
qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
)
print(qlayer.computation_space)

# Equivalent of no_bunching=False
qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.FOCK,
    ),
)
print(qlayer.computation_space)

ComputationSpace.UNBUNCHED
ComputationSpace.UNBUNCHED
ComputationSpace.FOCK


### 2.2 Constructor ``computation_space`` removed

The legacy no_bunching flag is no longer accepted on public layer and kernel entry points. It was deprecated in v.0.3 and is now removed in v.0.4.

Use explicit computation-space configuration instead:

In [ ]:
# Define the basic interferometer
circuit = ml.CircuitBuilder(n_modes=3)
circuit.add_entangling_layer()
circuit.add_angle_encoding([0, 1])
circuit.add_entangling_layer()

Old flag, breaking change

In [ ]:
# # Unbunched space
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     computation_space=ml.ComputationSpace.UNBUNCHED
#     )

# # Full Fock space
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     computation_space=ml.ComputationSpace.FOCK,
#     )

Equivalents of the flag

In [ ]:
# Equivalent of computation_space=ml.ComputationSpace.UNBUNCHED
qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.UNBUNCHED,
    ),
)
print(qlayer.computation_space)

# Equivalent of computation_space=ml.ComputationSpace.FOCK
qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.FOCK,
    ),
)
print(qlayer.computation_space)

ComputationSpace.UNBUNCHED
ComputationSpace.UNBUNCHED
ComputationSpace.FOCK


### 2.3 Legacy MeasurementStrategy access removed

Legacy enum-style measurement access such as MeasurementStrategy.PROBABILITIES, MeasurementStrategy.MODE_EXPECTATIONS, and MeasurementStrategy.AMPLITUDES now fails with a migration error. Passing measurement strategies as strings, such as "PROBABILITIES", is also no longer supported.

The accepted way is now to use ``MeasurementStrategy`` factory methods:

- ``MeasurementStrategy.probs(...)``
- ``MeasurementStrategy.mode_expectations(...)``
- ``MeasurementStrategy.amplitudes(...)``
- ``MeasurementStrategy.partial(...)``

Here are the suggested replacements

- ``MeasurementStrategy.PROBABILITIES``
    - ``MeasurementStrategy.probs(computation_space=...)``
- ``MeasurementStrategy.MODE_EXPECTATIONS``
    - ``MeasurementStrategy.mode_expectations(computation_space=...)``
- ``MeasurementStrategy.AMPLITUDES``
    - ``MeasurementStrategy.amplitudes(computation_space=...)``
- ``MeasurementStrategy.NONE``
    - ``MeasurementStrategy.amplitudes(computation_space=...)``
- ``"PROBABILITIES"`` (string)
    - ``MeasurementStrategy.probs(computation_space=...)``

We will present the migration for the probabilities strategy.

In [ ]:
# Define the basic interferometer
circuit = ml.CircuitBuilder(n_modes=3)
circuit.add_entangling_layer()
circuit.add_angle_encoding([0, 1])
circuit.add_entangling_layer()

Old accesses, breaking change

In [ ]:
# # Enum-style access
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     measurement_strategy=ml.MeasurementStrategy.PROBABILITIES
#     )

# # String access
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     measurement_strategy="PROBABILITIES"
#     )

Equivalents of the flag

In [ ]:
# Equivalent of no_bunching=True
qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(),
)
print(qlayer.measurement_strategy)

ComputationSpace.UNBUNCHED
ComputationSpace.UNBUNCHED
ComputationSpace.FOCK


### 2.4. Constructor amplitude encoding removed

``QuantumLayer(..., amplitude_encoding=True)`` is no longer accepted.

Amplitude inputs should now be passed to ``QuantumLayer.forward()`` as either a ``StateVector`` or a complex tensor.

Let's present these alternatives.

In [ ]:
# Define the basic interferometer
circuit = ml.CircuitBuilder(n_modes=3)
circuit.add_entangling_layer()
circuit.add_angle_encoding([0, 1])
circuit.add_entangling_layer()

Old flag, breaking change

In [ ]:
# # Enum-style access
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     amplitude_encoding=True
#     )

Equivalents of the flag

In [ ]:
# Option 1: StateVector input
input_state = ml.StateVector.from_tensor(
    tensor=torch.rand(1, 10),
    n_modes=4,
    n_photons=2,
    encoding=ml.EncodingSpace.FOCK,
)
layer(input_state)

# Option 2: complex tensor
input_state = ml.torch.rand(1, layer.output_size, dtype=torch.complex64)
layer(input_state)

### 2.5 Tensor constructor input_state removed

Passing a raw ``torch.Tensor`` as constructor input_state is no longer accepted. If a state object is needed at construction time, build it explicitly with ``StateVector.from_tensor(...)``.

Here is exactly how to migrate from the previous API.

In [ ]:
# Define the basic interferometer
circuit = ml.CircuitBuilder(n_modes=3)
circuit.add_entangling_layer()
circuit.add_angle_encoding([0, 1])
circuit.add_entangling_layer()

Old flag, breaking change

In [ ]:
# # Tensor input state
# amplitudes=torch.rand(2)
# qlayer=ml.QuantumLayer(
#     input_size=3,
#     builder=circuit,
#     n_photons=1,
#     input_state=amplitudes
#     )

Equivalents of the flag

In [ ]:
# StateVector input
amplitudes=torch.rand(3)
input_state=ml.StateVector.from_tensor(
      amplitudes,
      n_modes=3,
      n_photons=1,
      encoding=ml.EncodingSpace.UNBUNCHED,
  )

qlayer=ml.QuantumLayer(
    input_size=2,
    builder=circuit,n_photons=1,
    input_state=input_state
    )

## 3. Deprecations

Two new deprecations are effective since version v.0.4.

### 3.1 The ``remote_processor`` argument of the ``MerlinProcessor`` is deprecated

The name of the previous ``remote_processor`` argument is now simply ``processor`` where you pass the same Perceval processor.

In [ ]:
#Deprecated
# local_processor = pcvl.Processor("SLOS")
# proc = ml.MerlinProcessor(remote_processor=local_processor)

#New method
local_processor = pcvl.Processor("SLOS")
proc = ml.MerlinProcessor(processor=local_processor)

### 3.2 Kernel compatibility helpers are deprecated

Kernel compatibility helpers such as ``KernelCircuitBuilder``, ``FidelityKernel.simple(...)``, and related simple-factory compatibility arguments are deprecated and kept only for transition.

The kernel is now built on ``QuantumLayers``. No more compatibility are now necessary.

## 4. Compatibility and Maintenance

MerLin 0.4 updates the dependency floor to ``perceval-quandela>=1.2.1`` and allows newer ``PyTorch`` versions up to ``torch<=2.12.0``.

The test suite has been expanded substantially around noisy SLOS, g2 behavior, encoding spaces, state-vector inputs, deprecation removals, MerlinProcessor local execution, photonic generators, QCNNs, and QORC reservoir workflows.